In [ ]:
# Cell 1: Colab CPU setup. ARXIV_CONTACT is optional.
import os
import sys
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
REPO = Path("/content/intro-ds-arxiv-project")
BRANCH = "VA"
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", "https://github.com/thinhelzan842/intro-ds-arxiv-project.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "src/requirements.txt")], check=True)
sys.path.insert(0, str(REPO))
from src.download import run, assigned_ids, validate_assigned_manifest

DRIVE_ROOT = Path("/content/drive/MyDrive/arxiv-project")
OUTPUT = DRIVE_ROOT / "data/raw"
STATE = DRIVE_ROOT / "state/download"
MANIFEST = DRIVE_ROOT / "state/discovery_manifest.json"
STATE.mkdir(parents=True, exist_ok=True)
contact = os.environ.get("ARXIV_CONTACT", "").strip()
USER_AGENT = "HCMUS-IntroDS-arXiv-source/1.0 (HCMUS course project)"
if contact:
    if "@" not in contact:
        raise ValueError("ARXIV_CONTACT must be an email address when set.")
    USER_AGENT = f"HCMUS-IntroDS-arXiv-source/1.0 (mailto:{contact})"


In [ ]:
# Cell 2: discover once, retry unresolved IDs, then download/resume.
import json
import os
import uuid
import time

did_discover = False
if MANIFEST.is_file():
    saved = json.loads(MANIFEST.read_text(encoding="utf-8"))
    papers = saved["papers"] if isinstance(saved, dict) else saved
    expected = assigned_ids()
    by_id = {p["id"]: p for p in papers}
    if any(identifier not in expected for identifier in by_id):
        raise ValueError("Cached manifest contains IDs outside the assigned ranges")
    missing = [identifier for identifier in expected
               if identifier not in by_id or
               (by_id[identifier].get("status") == "error" and by_id[identifier].get("versions") == 0)]
    print(f"Loaded discovery manifest: {len(papers):,} IDs; missing/unresolved: {len(missing):,}")
else:
    papers = []
    missing = None

if missing is None or missing:
    previous = Path.cwd()
    os.chdir(REPO / "src")
    try:
        RUN_TEST = False
        get_ipython().run_line_magic("run", "-i ./01_discovery.ipynb")
        ids = all_assigned_ids() if missing is None else missing
        updates = fetch_metadata(ids)
        did_discover = True
        for paper in updates:
            save_metadata(paper)
    finally:
        os.chdir(previous)
    by_id = {p["id"]: p for p in papers}
    by_id.update({p["id"]: p for p in updates})
    papers = list(by_id.values())
    validate_assigned_manifest(papers)
    temporary = MANIFEST.with_name(MANIFEST.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        temporary.write_text(json.dumps({"papers": papers, "discovery_seconds": DISCOVERY_STATS["seconds"]},
                                        ensure_ascii=False), encoding="utf-8")
        os.replace(temporary, MANIFEST)
    finally:
        temporary.unlink(missing_ok=True)

validate_assigned_manifest(papers)
if did_discover:
    time.sleep(3)  # Keep the API-to-source transition at least 3 seconds apart.
summary = run(MANIFEST, OUTPUT, STATE, user_agent=USER_AGENT)
print(json.dumps(summary, indent=2, ensure_ascii=False))
